<a href="https://colab.research.google.com/github/xyosho/Proyecto-Homologacion-ML/blob/Sprint-3/sprint3_modelado_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sprint 3 — Modelado: la regresión logística del Sprint 2 frente a Random Forest y XGBoost
**Pedido del docente (Semana 3):** *un modelo de Machine Learning que ya haga predicciones* y una tabla comparativa
sencilla: *"El cálculo de la Semana 2 tenía un error del X%, mientras que este modelo tiene un error del Y%"*.

El proyecto predice **si el inscrito se matricula**. El Sprint 2 dejó una regresión logística sin ajustar; aquí se
ajustan sus hiperparámetros y se compara con dos modelos de árboles, Random Forest y XGBoost.

| Sección | Contenido |
|---|---|
| 1 | Variables de entrada (sin fuga de datos) |
| 2 | Ajuste de hiperparámetros con validación temporal |
| 3 | Comparación final en la prueba 2026-2 |
| 4 | Modelos finales y tabla de predicciones (bloque 3 del tablero) |

## Antes de empezar (Google Colab)
Las credenciales **no** se escriben en ninguna celda; se guardan en los **Secretos** de Colab:

1. Abra el panel **Secretos** (icono de llave, a la izquierda).
2. Cree `SUPABASE_URL` (la URL del proyecto, `https://….supabase.co`) y `SUPABASE_KEY` (la clave **publicable**,
   `sb_publishable_…`; nunca la secreta).
3. Active **Acceso del notebook** en los dos secretos.
4. Ejecute todo: *Entorno de ejecución → Ejecutar todas*. Si aparece un aviso para dar acceso a los secretos, acéptelo.

Las celdas que empiezan con `%%writefile` crean en Colab una copia exacta del código del repositorio (carpeta `src/`).

In [ ]:
!pip -q install supabase==2.33.0 xgboost==3.4.1 shap==0.54.1 lime==0.2.0.1

In [ ]:
%%writefile comun.py
"""Versión para Google Colab de src/comun.py: la configuración del proyecto (config/reglas_negocio.yaml)
viene incrustada porque Colab no tiene el repositorio. Generado por src/s06_construir_cuadernos.py."""
from pathlib import Path

RAIZ = Path.cwd()
DATA_PROCESSED = RAIZ / "data" / "processed"

_CONFIG = {'campus': {'nombre': 'Campus Unicervantes',
            'direccion': 'Calle 209 No. 104-15, vía Arrayanes, Bogotá',
            'lat': 4.790438,
            'lon': -74.066288,
            'fuente': 'Dirección: Acuerdo 002/2025 (membrete). Coordenada exacta de la placa Calle 209 # '
                      '104-15 (CL 209 104 15, lote 009140001001) en Placa Domiciliaria de Catastro Bogotá '
                      '(UAECD), consultada 23-oct-2026. Antes se usaba una coordenada aproximada de '
                      'OpenStreetMap (4.7895, -74.0660), a unos 100 m.'},
 'tarifas': {'AD': {2024: {'valor': 4419000,
                           'fuente': 'Precio de matrícula según recibos de pago: orden de pago Pregrado '
                                     "2024-2, concepto 'MAT ING ANALITICA DE DATOS NUEVOS'."},
                    2025: {'valor': 4658000,
                           'fuente': 'Precio de matrícula según recibos de pago 2025-1 y 2025-2: $3.260.600 '
                                     'con beca del 30% => 3.260.600 / 0,70; coincide con la versión 2025 de '
                                     'la página del programa.'},
                    2026: {'valor': 4914000,
                           'fuente': 'Acuerdo 002 del 11 de noviembre de 2025 del Consejo Directivo, art. 2 '
                                     '(valores de matrícula 2026).'},
                    2027: {'valor': 4914000,
                           'fuente': 'supuesto: S-01 (se mantiene el valor vigente hasta que se publique el '
                                     'acuerdo 2027).'}},
             'SSIA': {2026: {'valor': 4950000,
                             'fuente': 'Oferta académica oficial '
                                       'unicervantes.edu.co/admision/oferta-academica (consultada '
                                       '19-oct-2026).'},
                      2027: {'valor': 4950000, 'fuente': 'supuesto: S-01.'}}},
 'becas': {'SABER_11': {'nombre': 'Beca Saber 11',
                        'umbral_icfes': 310,
                        'pct_primer_periodo': 0.35,
                        'pct_siguientes': 0.3,
                        'promedio_minimo': 4.0,
                        'fuente': 'Admisiones (corrección del 31-oct-2026): 35% el 1er semestre y 30% desde '
                                  'el 2º; coincide con los recibos (2024-2: 35%; 2025 y 2026: 30%). Página '
                                  "oficial: 'Puntaje igual o superior a 310 ... 30% para toda la carrera por "
                                  "promedio académico'."},
           'MI_VOCACION': {'nombre': 'Beca Cervantes #MiVocación',
                           'pct_primer_periodo': 0.3,
                           'pct_siguientes': 0.2,
                           'promedio_minimo': 4.0,
                           'fuente': "Admisiones ('30% para el primer periodo académico y en adelante según "
                                     "la tabla relacionada'; se mantiene con promedio acumulado >= 4.0 y se "
                                     "pierde por sanciones); landing del programa ('30% 1er semestre y 20% "
                                     "el resto de la carrera')."},
           'NINGUNA': {'nombre': 'Sin beca',
                       'pct_primer_periodo': 0.0,
                       'pct_siguientes': 0.0,
                       'promedio_minimo': None},
           'acumulables': False},
 'particion': {'entrenamiento': ['2024-1', '2024-2', '2025-1', '2025-2'],
               'validacion': ['2026-1'],
               'prueba': ['2026-2'],
               'campana': ['2027-1']},
 'caracteristicas': {'bandas_icfes': [{'desde': 0, 'hasta': 250, 'segmento': 'BAJO'},
                                      {'desde': 250, 'hasta': 310, 'segmento': 'MEDIO'},
                                      {'desde': 310, 'hasta': 350, 'segmento': 'ALTO'},
                                      {'desde': 350, 'hasta': 501, 'segmento': 'SUPERIOR'}],
                     'zonas_distancia_km': {'CERCA': 10, 'MEDIA': 20},
                     'canal_agrupado': {'EVENTO': ['EXPOESTUDIANTES', 'CAMPUS_DAY'],
                                        'RELACIONAL': ['REFERIDO', 'INMERSION', 'COLEGIO_CONVENIO'],
                                        'DIGITAL': ['SITIO_WEB', 'REDES_SOCIALES', 'LLAMADA_WHATSAPP']},
                     'estrato_grupo': {1: 'BAJO', 2: 'BAJO', 3: 'MEDIO', 4: 'ALTO', 5: 'ALTO', 6: 'ALTO'},
                     'kmeans': {'k_min': 3,
                                'k_max': 7,
                                'tolerancia_silueta': 0.01,
                                'semilla': 20261029,
                                'variables': ['estrato',
                                              'log_distancia',
                                              'punt_global',
                                              'promedio_acumulado_colegio',
                                              'edad_al_inscribirse',
                                              'colegio_privado']}},
 'modelado': {'semilla': 20261105,
              'umbral_clasificacion': 0.5,
              'segmentos_propension': {'ALTA': 0.6, 'BAJA': 0.35},
              'sprint2_logistica': {'C': 1.0},
              'grillas': {'logistica': {'C': [0.01, 0.1, 1.0, 10.0]},
                          'random_forest': {'max_depth': [4, 6, 8, None],
                                            'min_samples_leaf': [5, 20],
                                            'max_features': ['sqrt', 0.5]},
                          'xgboost': {'n_estimators': [200, 400],
                                      'max_depth': [2, 3, 4],
                                      'learning_rate': [0.03, 0.1],
                                      'min_child_weight': [5]}}},
 'impacto': {'seguimiento': {'capacidad_pct': 0.3, 'efecto_maximo': 0.1, 'costo_por_inscrito': 30000},
             'expo': {'inversion_evento': 22000000}},
 'escala_desempeno_bachillerato': {'SUPERIOR': 4.6, 'ALTO': 4.0, 'BASICO': 3.0},
 'fondos_educacion_municipal': {'FOES_CHIA': {'ubicacion_id': 25175,
                                              'nombre': 'FOES - Fondo para el Fomento de la Educación '
                                                        'Superior de Chía',
                                              'norma': 'Acuerdo 172 de 2020; Resolución 2114 de 2021',
                                              'convenio_unicervantes': 'VERIFICADO',
                                              'apoyo_semestral_cop': 1000000,
                                              'requisitos': 'Residente de Chía (certificado de la JAC), '
                                                            'registro en el SISBÉN de Chía, certificado de '
                                                            'notas, Saber 11 y recibo de matrícula',
                                              'renovacion': 'Cada semestre (abril a octubre) según el '
                                                            'promedio del semestre anterior fijado en el '
                                                            'convenio',
                                              'fuente': 'Alcaldía de Chía (2022). Conozca el programa FOES: '
                                                        'Unicervantes en convenio (n.º 15); apoyo educativo '
                                                        'desde $1.000.000 en carreras presenciales.',
                                              'url': 'https://www.chia-cundinamarca.gov.co/2022/educacion/PROGRAMA%20FOES%20ACTUALIZADO%2022-09-2022.pdf'},
                                'FOES_SOPO': {'ubicacion_id': 25758,
                                              'nombre': 'FOES - Fondo Especial para el Fomento de la '
                                                        'Educación de Sopó',
                                              'norma': 'No publicada en la página consultada',
                                              'convenio_unicervantes': 'NO_VERIFICADO',
                                              'apoyo_semestral_cop': 1000000,
                                              'requisitos': 'Residente de Sopó, admitido o matriculado en un '
                                                            'programa de educación superior',
                                              'renovacion': 'No publicada',
                                              'fuente': 'Alcaldía de Sopó (2022, actualizado 2026). FOES: '
                                                        'apoyo de matrícula, transporte y sostenimiento. '
                                                        'Monto: supuesto S-31.',
                                              'url': 'https://www.sopo-cundinamarca.gov.co/educacion/publicaciones/290/foes-fondo-especial-para-el-fomento-de-la-educacion/'},
                                'FOEST_TOCANCIPA': {'ubicacion_id': 25817,
                                                    'nombre': 'FOEST - Fondo Especial de Educación Superior '
                                                              'de Tocancipá',
                                                    'norma': 'Aprobado por el Concejo Municipal (acuerdo no '
                                                             'consultado)',
                                                    'convenio_unicervantes': 'NO_VERIFICADO',
                                                    'apoyo_semestral_cop': 1000000,
                                                    'requisitos': 'Residente de Tocancipá; becas para '
                                                                  'mejores puntajes Saber 11, artistas y '
                                                                  'deportistas; descuentos por convenio',
                                                    'renovacion': 'No publicada',
                                                    'fuente': 'Alcaldía de Tocancipá (2017). $1.750 millones '
                                                              'para la educación superior. Monto por '
                                                              'estudiante: supuesto S-31.',
                                                    'url': 'https://www.tocancipa-cundinamarca.gov.co/sala-prensa/noticias-1/1-750-millones-para-educacion-superior-tocancipa'}},
 'inversion_captacion': {'por_evento': {'EXPOESTUDIANTES': 22000000, 'CAMPUS_DAY': 6000000},
                         'por_periodo': {'SITIO_WEB': {'1': 9000000, '2': 6000000},
                                         'REDES_SOCIALES': {'1': 18000000, '2': 10000000},
                                         'LLAMADA_WHATSAPP': {'1': 8000000, '2': 6000000},
                                         'COLEGIO_CONVENIO': {'1': 7000000, '2': 4000000},
                                         'INMERSION': {'1': 5000000, '2': 5000000},
                                         'REFERIDO': {'1': 3000000, '2': 3000000}},
                         'ventana_campana': {'1': ['03-01', '01-25'], '2': ['01-10', '07-25']}},
 'generador': {'matricula': {'tasa_objetivo': 0.45,
                             'beta_descuento': 3.0,
                             'beta_estrato': {1: -0.5, 2: -0.15, 3: 0.0, 4: 0.1, 5: 0.0, 6: -0.3},
                             'beta_log_distancia': -0.35,
                             'beta_icfes_cuadratico': -0.25,
                             'beta_canal': {'EXPOESTUDIANTES': -0.3,
                                            'SITIO_WEB': 0.0,
                                            'REDES_SOCIALES': -0.1,
                                            'REFERIDO': 0.5,
                                            'CAMPUS_DAY': 0.3,
                                            'LLAMADA_WHATSAPP': -0.2,
                                            'INMERSION': 0.8,
                                            'COLEGIO_CONVENIO': 0.3},
                             'beta_entrevista': 0.6,
                             'beta_programa_ssia': 0.1,
                             'beta_periodo_2': -0.1,
                             'beta_foes': 0.5}}}


def cargar_config() -> dict:
    return _CONFIG


In [ ]:
%%writefile conexion_supabase.py
"""Conexión SEGURA de Python con Supabase para leer la tabla plana dataset_matricula.

Las credenciales NUNCA se escriben en el código ni se imprimen:
  - En el computador: archivo .env en la raíz del proyecto, excluido por .gitignore (plantilla: .env.example).
    Se puede crear con:  python src/s00_configurar_credenciales.py
  - En Google Colab: panel "Secretos" (icono de llave) con SUPABASE_URL y SUPABASE_KEY, con acceso al cuaderno.

SUPABASE_KEY es la clave PUBLICABLE del proyecto. En la base, la seguridad a nivel de fila (RLS) solo le permite
LEER dataset_matricula (sin datos personales); las otras 25 tablas están cerradas y nadie puede escribir con ella.
Así, aunque la clave se filtrara, solo se podría leer el dataset anonimizado.
"""
from __future__ import annotations

import os
from pathlib import Path

import pandas as pd

TABLA = "dataset_matricula"
VARIABLES = ("SUPABASE_URL", "SUPABASE_KEY")
TAMANO_PAGINA = 1000          # la API de Supabase entrega como máximo 1.000 filas por petición
TIEMPO_LIMITE_S = 30          # si la red no responde, se desiste en vez de quedarse esperando


def en_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


def obtener_credenciales() -> tuple[str, str]:
    """Devuelve (url, clave) desde los Secretos de Colab o desde .env / variables de entorno.

    En Colab, la primera vez que un cuaderno lee un secreto aparece un aviso para darle acceso: hay que aceptarlo
    (mientras no se acepte, la celda espera). Los errores explican qué falta, nunca muestran los valores.
    """
    problemas = []
    if en_colab():
        from google.colab import userdata
        valores = {}
        for v in VARIABLES:
            try:
                valores[v] = (userdata.get(v) or "").strip()
            except Exception as e:                  # SecretNotFoundError / NotebookAccessError
                valores[v] = ""
                motivo = ("el cuaderno no tiene acceso: active 'Acceso del notebook'" if "Access" in type(e).__name__
                          else "no existe un secreto con ese nombre exacto")
                problemas.append(f"{v}: {motivo}")
        donde = "los Secretos de Colab (icono de llave, con 'Acceso del notebook' activado)"
    else:
        from dotenv import load_dotenv
        load_dotenv(Path(__file__).resolve().parents[1] / ".env")
        valores = {v: os.environ.get(v, "").strip() for v in VARIABLES}
        donde = "el archivo .env (python src/s00_configurar_credenciales.py o copia de .env.example)"
    problemas += [f"{v}: está vacío" for v, x in valores.items() if not x and not any(p.startswith(v) for p in problemas)]
    url, clave = valores["SUPABASE_URL"], valores["SUPABASE_KEY"]
    if url and not (url.startswith("https://") and url.rstrip("/").endswith(".supabase.co")):
        problemas.append("SUPABASE_URL: no es la URL del proyecto (debe ser https://<proyecto>.supabase.co; "
                         "no la cadena postgresql:// del pooler)")
    if clave.lower().startswith(("postgres://", "postgresql://")):
        problemas.append("SUPABASE_KEY: es la cadena del pooler; aquí va la clave PUBLICABLE (sb_publishable_...)")
    elif clave.startswith("sb_secret_"):
        problemas.append("SUPABASE_KEY: es la clave SECRETA; use la PUBLICABLE (sb_publishable_...)")
    if problemas:
        raise RuntimeError(f"Credenciales incompletas en {donde}: " + "; ".join(problemas) + ".")
    return url, clave


def crear_cliente(url: str | None = None, clave: str | None = None):
    """Cliente de la API de Supabase con tiempo límite. La clave se usa en memoria y no se muestra."""
    from supabase import ClientOptions, create_client
    if url is None or clave is None:
        url, clave = obtener_credenciales()
    return create_client(url, clave, options=ClientOptions(postgrest_client_timeout=TIEMPO_LIMITE_S))


def leer_dataset_matricula(cliente, tabla: str = TABLA) -> pd.DataFrame:
    """Lee la tabla completa en páginas de 1.000 filas (límite de la API), ordenada por inscripcion_id."""
    filas, inicio = [], 0
    while True:
        r = (cliente.table(tabla).select("*").order("inscripcion_id")
             .range(inicio, inicio + TAMANO_PAGINA - 1).execute())
        filas += r.data
        if len(r.data) < TAMANO_PAGINA:
            break
        inicio += TAMANO_PAGINA
    return pd.DataFrame(filas)


def probar_solo_lectura(cliente, tabla: str = TABLA) -> bool:
    """Intenta modificar la tabla con la clave publicable. Devuelve True si la base lo RECHAZA por permisos
    (código 42501 de PostgreSQL: permiso denegado). La fila -1 no existe, así que nada cambiaría de todos modos."""
    try:
        cliente.table(tabla).update({"punt_global": 0}).eq("inscripcion_id", -1).execute()
    except Exception as e:                           # APIError de la API de Supabase
        return getattr(e, "code", None) == "42501"
    return False


In [ ]:
%%writefile datos.py
"""Carga de las tablas planas para los sprints 2 a 6.

Fuente preferida: Supabase con la API de solo lectura (credenciales en .env o en los Secretos de Colab, ver
conexion_supabase.py). Si no hay credenciales, se usan los CSV que exporta el pipeline (data/processed), que son
idénticos (las pruebas lo comprueban). La función siempre dice de dónde salieron los datos.
"""
from __future__ import annotations

from pathlib import Path

import pandas as pd

TABLAS = ("dataset_matricula", "dataset_entrevista", "resultado_campana_2027_1")   # el último solo en el Sprint 5
ENTEROS_CON_VACIOS = ("matriculado", "puntaje_vocacional", "matriculado_real", "entrevista_interes_programa",
                      "entrevista_proyecto_vida", "entrevista_interes_universidad", "entrevista_conciencia_social",
                      "entrevista_expresion")


def preparar_tipos(df: pd.DataFrame) -> pd.DataFrame:
    """La API devuelve JSON: se fijan los tipos para que local y Colab den exactamente lo mismo."""
    if df.empty:                                   # tabla sin filas: no hay tipos que fijar
        return df
    df = df.copy()
    for c in ENTEROS_CON_VACIOS:
        if c in df:
            df[c] = pd.to_numeric(df[c]).astype("Int64")
    for c in ("distancia_campus_km", "distancia_vial_km", "colegio_promedio_saber11", "promedio_acumulado_colegio",
              "promedio_matematicas_colegio", "pct_descuento_inicial"):
        if c in df:
            df[c] = pd.to_numeric(df[c]).astype(float)
    if "asesor_id" in df:
        df["asesor_id"] = df["asesor_id"].astype(int)
    return df.sort_values("inscripcion_id").reset_index(drop=True)


def _csv_local(tabla: str) -> Path:
    from comun import DATA_PROCESSED
    if tabla == "resultado_campana_2027_1":       # "sobre sellado" desde la Semana 1 (data/processed/oculto)
        return DATA_PROCESSED / "oculto" / "resultado_2027_1_sellado.parquet"
    return DATA_PROCESSED / f"{tabla}.csv"


def cargar(tabla: str, fuente: str = "auto") -> tuple[pd.DataFrame, str]:
    """Devuelve (datos, origen). fuente: 'supabase', 'local' o 'auto' (Supabase si hay credenciales)."""
    if tabla not in TABLAS:
        raise ValueError(f"Tabla desconocida: {tabla}")
    if fuente in ("auto", "supabase"):
        try:
            from conexion_supabase import crear_cliente, leer_dataset_matricula
            df = leer_dataset_matricula(crear_cliente(), tabla=tabla)
            return preparar_tipos(df), "Supabase (API de solo lectura)"
        except Exception as error:
            if fuente == "supabase":
                raise
            aviso = type(error).__name__
    ruta = _csv_local(tabla)
    df = pd.read_parquet(ruta) if ruta.suffix == ".parquet" else pd.read_csv(ruta)
    origen = "CSV local exportado por el pipeline" + (f" (sin conexión a Supabase: {aviso})" if fuente == "auto" else "")
    return preparar_tipos(df), origen


def cargar_inscritos(fuente: str = "auto") -> tuple[pd.DataFrame, str]:
    """dataset_matricula con la rúbrica de la entrevista (dataset_entrevista) unida por inscripcion_id."""
    inscritos, origen = cargar("dataset_matricula", fuente)
    entrevista, _ = cargar("dataset_entrevista", fuente)
    if len(entrevista) != len(inscritos):     # p. ej. la tabla existe en Supabase pero aún no se cargó (src/s05)
        if fuente == "supabase":
            raise RuntimeError(f"dataset_entrevista tiene {len(entrevista)} filas y dataset_matricula {len(inscritos)}: "
                               "falta cargar la entrevista en Supabase (python src/s05_cargar_supabase.py).")
        entrevista, _ = cargar("dataset_entrevista", "local")
    return inscritos.merge(entrevista, on="inscripcion_id", how="left", validate="one_to_one"), origen


In [ ]:
%%writefile caracteristicas.py
"""SPRINT 2 — Ingeniería de características: columnas nuevas y útiles sobre la tabla plana dataset_matricula.

Regla de oro contra la FUGA DE DATOS: todo lo que se APRENDE de los datos (las medias para estandarizar y los grupos
de k-means) se ajusta SOLO con el conjunto de ENTRENAMIENTO
(2024-1 a 2025-2) y después se aplica igual a validación, prueba, campaña y a cualquier aspirante nuevo (Sprint 6).
Si se ajustara con todos los datos, el modelo "vería" información del futuro y sus métricas serían optimistas.

Funciones principales:
  particion_temporal(periodo)                      -> ENTRENAMIENTO / VALIDACION / PRUEBA / CAMPANA
  ajustar_referencias(inscritos)                   -> lo aprendido del entrenamiento (un diccionario)
  construir_caracteristicas(inscritos, referencias) -> la misma tabla con las columnas nuevas
"""
from __future__ import annotations

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

NUEVAS = [
    "particion", "aplica_beca_saber11", "aplica_beca_mi_vocacion", "indice_vocacional",
    "segmento_rendimiento_icfes", "brecha_colegio_vs_icfes",
    "ahorro_beca_primer_semestre", "log_distancia", "zona_distancia", "rezago_saber11",
    "canal_agrupado", "estrato_grupo", "colegio_privado", "segmento_mercado",
]

# Las 5 secciones de la entrevista de admisión calificadas de 1 a 5 (dataset_entrevista, supuesto S-46)
DIMENSIONES_ENTREVISTA = ["entrevista_interes_programa", "entrevista_proyecto_vida", "entrevista_interes_universidad",
                          "entrevista_conciencia_social", "entrevista_expresion"]

# Descripción de cada columna nueva (diccionario del Sprint 2)
DICCIONARIO = {
    "particion": "Conjunto temporal: ENTRENAMIENTO (2024-1 a 2025-2), VALIDACION (2026-1), PRUEBA (2026-2), CAMPANA (2027-1)",
    "aplica_beca_saber11": "Sí/No: puntaje global Saber 11 ≥ 310 (requisito de la Beca Saber 11)",
    "aplica_beca_mi_vocacion": "Sí/No: presentó entrevista y su puntaje vocacional es ≥ 3 (requisito de Mi Vocación)",
    "indice_vocacional": "Índice de 0 a 100 de la entrevista de admisión: 25 × (promedio de la rúbrica 1-5 de sus 5 secciones − 1); vacío si no hubo entrevista",
    "segmento_rendimiento_icfes": "BAJO (< 250), MEDIO (250-309), ALTO (310-349: ya aplica a beca) o SUPERIOR (≥ 350)",
    "brecha_colegio_vs_icfes": "z(promedio del colegio) − z(Saber 11): positivo = rinde más en el colegio que en el examen",
    "ahorro_beca_primer_semestre": "Cuánto le ahorra la beca en el 1er semestre (tarifa plena − precio neto), en COP",
    "log_distancia": "Logaritmo natural de la distancia por vías desde la dirección hasta el campus (S-44; reduce la cola larga)",
    "zona_distancia": "Distancia por vías desde la dirección: CERCA (≤ 10 km), MEDIA (10-20 km) o LEJOS (> 20 km)",
    "rezago_saber11": "INMEDIATO (presentó el Saber 11 el año anterior), UN_ANIO o DOS_O_MAS",
    "canal_agrupado": "EVENTO (Expo, Campus Day), RELACIONAL (referido, inmersión, convenio) o DIGITAL (web, redes, llamada)",
    "estrato_grupo": "BAJO (1-2), MEDIO (3) o ALTO (4-6)",
    "colegio_privado": "1 si el colegio es privado",
    "segmento_mercado": "Perfil del aspirante según k-means (segmentación de mercado), nombrado por sus rasgos principales",
}

ETIQUETAS_KMEANS = {   # (si el centro está por encima, si está por debajo) de la media del entrenamiento
    "estrato": ("estrato más alto", "estrato más bajo"),
    "log_distancia": ("viven lejos del campus", "viven cerca del campus"),
    "punt_global": ("buen puntaje ICFES", "puntaje ICFES bajo"),
    "promedio_acumulado_colegio": ("buenas notas en el colegio", "notas bajas en el colegio"),
    "edad_al_inscribirse": ("aspirantes mayores", "aspirantes más jóvenes"),
    "colegio_privado": ("colegio privado", "colegio público"),
    "colegio_promedio_saber11": ("colegio de alto nivel", "colegio de nivel bajo"),
}

# Nombres en español sencillo para tablas y gráficas (los nombres técnicos de las columnas no cambian)
NOMBRES_LEGIBLES = {
    "estrato": "Estrato promedio",
    "log_distancia": "Distancia por vías (km)",
    "distancia_vial_km": "Distancia por vías (km)",
    "punt_global": "Puntaje ICFES",
    "promedio_acumulado_colegio": "Notas del colegio (1 a 5)",
    "edad_al_inscribirse": "Edad (años)",
    "colegio_privado": "% de colegio privado",
    "colegio_promedio_saber11": "Nivel ICFES del colegio",
    "segmento_mercado": "Segmento de mercado",
    "segmento_rendimiento_icfes": "Puntaje ICFES",
    "aplica_beca_mi_vocacion": "¿Aplica a la beca Mi Vocación?",
    "aplica_beca_saber11": "¿Aplica a la beca Saber 11?",
    "canal_agrupado": "Canal por el que llegó",
    "zona_distancia": "Distancia al campus",
    "estrato_grupo": "Estrato",
    "rezago_saber11": "Tiempo desde el ICFES",
    "indice_vocacional": "Índice vocacional de la entrevista (0 a 100)",
}

# El ICFES llama "oficial" al colegio público. Los datos y los modelos conservan el código OFICIAL; en todo texto que
# se muestra (explicaciones, gráficas, script de predicción) se dice "público".
NOMBRE_TIPO_COLEGIO = {"OFICIAL": "público", "PRIVADO": "privado", "TECNICO": "técnico"}


def tipo_colegio_codigo(valor: str) -> str:
    """Código de datos del tipo de colegio: acepta PUBLICO (o PÚBLICO) como sinónimo de OFICIAL."""
    v = str(valor).strip().upper().replace("Ú", "U").replace("É", "E")
    return "OFICIAL" if v == "PUBLICO" else v


def etiquetas_categorias(cfg: dict | None = None) -> dict:
    """Texto claro para los códigos de las columnas categóricas nuevas (p. ej. CERCA → "Cerca (hasta 10 km)")."""
    c = (cfg or configuracion())["caracteristicas"]
    z = c["zonas_distancia_km"]
    icfes = {b["segmento"]: (f"{b['segmento'].capitalize()} (menos de {b['hasta']})" if b["desde"] == 0 else
                             f"{b['segmento'].capitalize()} ({b['desde']} o más)" if b["hasta"] > 500 else
                             f"{b['segmento'].capitalize()} ({b['desde']} a {b['hasta'] - 1})") for b in c["bandas_icfes"]}
    return {
        "segmento_rendimiento_icfes": icfes,
        "zona_distancia": {"CERCA": f"Cerca (hasta {z['CERCA']} km)", "MEDIA": f"Media ({z['CERCA']} a {z['MEDIA']} km)",
                           "LEJOS": f"Lejos (más de {z['MEDIA']} km)"},
        "canal_agrupado": {"EVENTO": "Eventos (Expo, Campus Day)", "RELACIONAL": "Relacional (referido, convenio…)",
                           "DIGITAL": "Digital (web, redes, llamada)"},
        "estrato_grupo": {"BAJO": "Estratos 1 y 2", "MEDIO": "Estrato 3", "ALTO": "Estratos 4 a 6"},
        "rezago_saber11": {"INMEDIATO": "Sin pausa tras el colegio", "UN_ANIO": "1 año de pausa", "DOS_O_MAS": "2 años o más de pausa"},
        "aplica_beca_mi_vocacion": {"Sí": "Sí aplica", "No": "No aplica"},
        "aplica_beca_saber11": {"Sí": "Sí aplica", "No": "No aplica"},
    }


def tabla_segmentos(d: pd.DataFrame) -> pd.DataFrame:
    """Resumen de cada segmento de mercado con valores reales (no estandarizados), listo para leer o presentar."""
    m = d.assign(matriculado=d["matriculado"].astype(float))
    def resumen(g: pd.DataFrame) -> dict:
        return {"Inscritos": len(g), "% del total": len(g) / len(d) * 100,
                "% que se matriculó": g["matriculado"].mean() * 100, "Puntaje ICFES": g["punt_global"].mean(),
                "Notas del colegio (1 a 5)": g["promedio_acumulado_colegio"].mean(),
                "Distancia por vías (km)": g["distancia_vial_km"].mean(), "Estrato promedio": g["estrato"].mean(),
                "% de colegio público": (1 - g["colegio_privado"].mean()) * 100, "Edad (años)": g["edad_al_inscribirse"].mean()}
    t = pd.DataFrame({s: resumen(g) for s, g in m.groupby("segmento_mercado")}).T
    t.loc["Todos los inscritos"] = resumen(m)
    t.index.name = "Segmento de mercado"
    return t.round({"Inscritos": 0, "% del total": 1, "% que se matriculó": 1, "Puntaje ICFES": 0,
                    "Notas del colegio (1 a 5)": 2, "Distancia por vías (km)": 1, "Estrato promedio": 1,
                    "% de colegio público": 0, "Edad (años)": 1}).astype({"Inscritos": int})


def configuracion() -> dict:
    """Parámetros del proyecto: config/reglas_negocio.yaml en el computador; en Colab, la copia del cuaderno."""
    try:
        from comun import cargar_config
        return cargar_config()
    except ImportError:
        return CONFIG_PROYECTO  # noqa: F821  (lo define una celda del cuaderno de Colab)


# ----------------------------------------------------------------------------- reglas (no se aprenden de datos)
def particion_temporal(periodo: pd.Series, cfg: dict | None = None) -> pd.Series:
    p = (cfg or configuracion())["particion"]
    mapa = {per: nombre.upper() for nombre in ("entrenamiento", "validacion", "prueba", "campana") for per in p[nombre]}
    return periodo.map(mapa)


def segmento_icfes(punt_global: pd.Series, cfg: dict | None = None) -> pd.Series:
    bandas = (cfg or configuracion())["caracteristicas"]["bandas_icfes"]
    cortes = [bandas[0]["desde"]] + [b["hasta"] for b in bandas]
    return pd.cut(punt_global, bins=cortes, right=False, labels=[b["segmento"] for b in bandas]).astype(str)


def indice_vocacional(df: pd.DataFrame) -> pd.Series:
    """Entrevista cualitativa → número: 25 × (promedio de las 5 secciones calificadas de 1 a 5 − 1), de 0 a 100.
    Vacío si no hubo entrevista. Es una regla fija (no se aprende de los datos), así que no hay riesgo de fuga."""
    return (25 * (df[DIMENSIONES_ENTREVISTA].astype(float).mean(axis=1) - 1)).round(1)


def _reglas(df: pd.DataFrame, cfg: dict) -> pd.DataFrame:
    c = cfg["caracteristicas"]
    d = df.copy()
    d["particion"] = particion_temporal(d["periodo"], cfg)
    d["aplica_beca_saber11"] = np.where(d["punt_global"] >= cfg["becas"]["SABER_11"]["umbral_icfes"], "Sí", "No")
    d["aplica_beca_mi_vocacion"] = np.where(d["cumple_req_mi_vocacion"] == 1, "Sí", "No")
    d["indice_vocacional"] = indice_vocacional(d)
    d["segmento_rendimiento_icfes"] = segmento_icfes(d["punt_global"], cfg)
    # El precio que se usa es el neto del 1er semestre (valor_neto_primer_semestre, ya en dataset_matricula)
    d["ahorro_beca_primer_semestre"] = d["valor_semestre_bruto"] - d["valor_neto_primer_semestre"]
    vial = d["distancia_vial_km"].astype(float)                 # desde la dirección del formulario (S-44)
    d["log_distancia"] = np.log(np.maximum(vial, 0.5))           # piso de 0,5 km: un vecino del campus no da −∞
    z = c["zonas_distancia_km"]
    d["zona_distancia"] = np.select([vial <= z["CERCA"], vial <= z["MEDIA"]],
                                    ["CERCA", "MEDIA"], "LEJOS")
    d["rezago_saber11"] = np.select([d["anios_desde_saber11"] == 0, d["anios_desde_saber11"] == 1],
                                    ["INMEDIATO", "UN_ANIO"], "DOS_O_MAS")
    canal = {canal: grupo for grupo, canales in c["canal_agrupado"].items() for canal in canales}
    d["canal_agrupado"] = d["canal"].map(canal)
    d["estrato_grupo"] = d["estrato"].map(lambda e: c["estrato_grupo"][int(e)])
    d["colegio_privado"] = (d["colegio_naturaleza"] == "PRIVADO").astype(int)
    return d


# ----------------------------------------------------------------------------- lo que se aprende del entrenamiento
def ajustar_referencias(inscritos: pd.DataFrame, cfg: dict | None = None) -> dict:
    """Aprende, SOLO con las filas de ENTRENAMIENTO, todo lo que necesitan las columnas nuevas."""
    cfg = cfg or configuracion()
    ent = _reglas(inscritos, cfg)
    ent = ent[ent["particion"] == "ENTRENAMIENTO"]
    ref: dict = {"periodos_entrenamiento": sorted(ent["periodo"].unique()), "n_entrenamiento": len(ent)}

    # 1) Estandarización (z) con medias y desviaciones del entrenamiento
    ref["estandarizacion"] = {c: (float(ent[c].mean()), float(ent[c].std())) for c in ["punt_global", "promedio_acumulado_colegio"]}

    # 2) Segmentación de mercado con k-means: se prueba k = k_min..k_max; se elige el MENOR k cuya silueta esté a
    #    menos de tolerancia_silueta de la mejor (parsimonia: menos segmentos son más fáciles de usar para Mercadeo)
    km = cfg["caracteristicas"]["kmeans"]
    variables = km["variables"]
    medias, desv = ent[variables].mean(), ent[variables].std()
    Z = ((ent[variables] - medias) / desv).to_numpy(float)
    evaluacion = []
    for k in range(km["k_min"], km["k_max"] + 1):
        modelo = KMeans(n_clusters=k, n_init=20, random_state=km["semilla"]).fit(Z)
        evaluacion.append({"k": k, "inercia": float(modelo.inertia_),
                           "silueta": float(silhouette_score(Z, modelo.labels_, random_state=km["semilla"]))})
    tope = max(e["silueta"] for e in evaluacion) - km.get("tolerancia_silueta", 0.0)
    mejor = min((e for e in evaluacion if e["silueta"] >= tope), key=lambda e: e["k"])
    modelo = KMeans(n_clusters=mejor["k"], n_init=20, random_state=km["semilla"]).fit(Z)
    orden = np.argsort(-np.bincount(modelo.labels_))            # segmento 1 = el más grande
    centros = pd.DataFrame(modelo.cluster_centers_[orden], columns=variables)
    nombres = []
    for i, fila in centros.iterrows():
        rasgos = fila.abs().sort_values(ascending=False).index[:2]
        rasgos_txt = ", ".join(ETIQUETAS_KMEANS[v][0 if fila[v] > 0 else 1] for v in rasgos)
        nombres.append(f"S{i + 1}: " + rasgos_txt[0].upper() + rasgos_txt[1:])   # "S1: Colegio público, viven cerca…"
    ref["kmeans"] = {"modelo": modelo, "orden": orden.tolist(), "variables": variables, "medias": medias, "desv": desv,
                     "evaluacion": evaluacion, "k": mejor["k"], "nombres": nombres, "centros_z": centros}
    return ref


def construir_caracteristicas(inscritos: pd.DataFrame, ref: dict, cfg: dict | None = None) -> pd.DataFrame:
    """Devuelve la misma tabla con las columnas nuevas, aplicando lo aprendido del entrenamiento."""
    cfg = cfg or configuracion()
    d = _reglas(inscritos, cfg)
    (mg, sg), (mc, sc) = ref["estandarizacion"]["punt_global"], ref["estandarizacion"]["promedio_acumulado_colegio"]
    d["brecha_colegio_vs_icfes"] = (((d["promedio_acumulado_colegio"] - mc) / sc) - ((d["punt_global"] - mg) / sg)).round(3)
    km = ref["kmeans"]
    Z = ((d[km["variables"]] - km["medias"]) / km["desv"]).to_numpy(float)
    etiqueta = km["modelo"].predict(Z)
    posicion = {original: nuevo for nuevo, original in enumerate(km["orden"])}
    d["segmento_mercado"] = [km["nombres"][posicion[e]] for e in etiqueta]
    return d[list(inscritos.columns) + [c for c in NUEVAS if c not in inscritos.columns]]


In [ ]:
%%writefile modelado.py
"""Modelos y métricas de la matrícula (Sprint 2: regresión logística · Sprint 3: comparación con Random Forest y XGBoost).

El primer modelo del proyecto (Sprint 2) es una REGRESIÓN LOGÍSTICA sin ajuste de hiperparámetros
(config/reglas_negocio.yaml -> modelado.sprint2_logistica): es el modelo base del proyecto. En el Sprint 3 se ajustan
sus hiperparámetros con la validación y se compara con Random Forest y XGBoost.
"""
from __future__ import annotations

import numpy as np
import pandas as pd
from sklearn.metrics import (average_precision_score, brier_score_loss, f1_score, log_loss, precision_score,
                             recall_score, roc_auc_score)


def metricas(y, p, umbral: float = 0.5) -> dict:
    """Métricas de clasificación. 'error_pct' es el lenguaje del docente: % de inscritos mal clasificados."""
    y = np.asarray(y, dtype=int)
    p = np.clip(np.asarray(p, dtype=float), 1e-6, 1 - 1e-6)
    pred = (p >= umbral).astype(int)
    unica = len(np.unique(y)) > 1
    return {
        "n": int(len(y)), "umbral": float(umbral),
        "error_pct": float(100 * (pred != y).mean()),
        "exactitud": float((pred == y).mean()),
        "auc_roc": float(roc_auc_score(y, p)) if unica else float("nan"),
        "auc_pr": float(average_precision_score(y, p)) if unica else float("nan"),
        "brier": float(brier_score_loss(y, p)),
        "log_loss": float(log_loss(y, p, labels=[0, 1])),
        "precision": float(precision_score(y, pred, zero_division=0)),
        "recall": float(recall_score(y, pred, zero_division=0)),
        "f1": float(f1_score(y, pred, zero_division=0)),
        "tasa_real": float(y.mean()), "tasa_predicha_media": float(p.mean()),
    }


def tabla_metricas(resultados: dict[str, dict]) -> pd.DataFrame:
    """{nombre_modelo: metricas} -> tabla ordenada para mostrar."""
    cols = ["n", "error_pct", "exactitud", "auc_roc", "auc_pr", "brier", "precision", "recall", "f1"]
    return pd.DataFrame(resultados).T[cols].astype(float).round(4)


# =============================================================================
# Modelos de Machine Learning (regresión logística desde el Sprint 2; Random Forest y XGBoost en el Sprint 3)
# =============================================================================
# Variables de entrada. Todas se conocen ANTES de la decisión que se predice (sin fuga de datos).
# No se usa colegio_promedio_saber11 (media ICFES del colegio): Admisiones no suele tener ese dato (bitácora H-43).
VARIABLES_MATRICULA = {
    "numericas": ["punt_global", "icfes_al_cuadrado", "promedio_acumulado_colegio", "edad_al_inscribirse", "estrato",
                  "log_distancia", "dias_registro_a_inscripcion", "contactos_antes_de_inscribirse",
                  "anios_desde_saber11", "indice_vocacional", "pct_descuento_inicial",
                  "valor_neto_primer_semestre", "brecha_colegio_vs_icfes"],
    "binarias": ["entrevista_realizada", "apoyo_foes", "elegible_foes", "colegio_en_convenio", "captado_en_evento",
                 "colegio_privado", "semestre_2"],
    "categoricas": ["programa", "genero", "canal", "beca", "tipo_colegio", "regimen_salud", "segmento_mercado"],
}


def preparar_X(d: pd.DataFrame, referencias: dict, variables: dict) -> pd.DataFrame:
    """Columnas de entrada del modelo, con el tratamiento de nulos del Sprint 1 y dos derivadas simples."""
    X = d.copy()
    X["indice_vocacional"] = X["indice_vocacional"].astype("Float64").fillna(0).astype(float)   # 0 = sin entrevista
    # (entrevista_realizada distingue "sin entrevista" de un índice de 0)
    media, desv = referencias["estandarizacion"]["punt_global"]
    X["icfes_al_cuadrado"] = ((X["punt_global"] - media) / desv) ** 2       # captura la "U invertida" del ICFES
    X["semestre_2"] = X["periodo"].str.endswith("-2").astype(int)          # ingreso a mitad de año
    columnas = variables["numericas"] + variables["binarias"] + variables["categoricas"]
    X = X[columnas].copy()
    for c in variables["categoricas"]:
        X[c] = X[c].astype(str)
    for c in variables["numericas"] + variables["binarias"]:
        X[c] = X[c].astype(float)
    return X


def construir_pipeline(tipo: str, parametros: dict, variables: dict, semilla: int = 20261105):
    """Preprocesamiento + modelo en un solo objeto: se ajusta y se aplica igual en todos los conjuntos."""
    from sklearn.compose import ColumnTransformer
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import OneHotEncoder, StandardScaler
    from xgboost import XGBClassifier

    categ = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    if tipo == "logistica":
        prep = ColumnTransformer([("num", StandardScaler(), variables["numericas"] + variables["binarias"]),
                                  ("cat", categ, variables["categoricas"])], verbose_feature_names_out=False)
        modelo = LogisticRegression(max_iter=5000, **parametros)
    else:
        prep = ColumnTransformer([("num", "passthrough", variables["numericas"] + variables["binarias"]),
                                  ("cat", categ, variables["categoricas"])], verbose_feature_names_out=False)
        if tipo == "random_forest":
            modelo = RandomForestClassifier(n_estimators=500, random_state=semilla, n_jobs=-1, **parametros)
        elif tipo == "xgboost":
            modelo = XGBClassifier(random_state=semilla, n_jobs=1, eval_metric="logloss", tree_method="hist",
                                   subsample=0.8, colsample_bytree=0.8, **parametros)
        else:
            raise ValueError(tipo)
    return Pipeline([("preparacion", prep), ("modelo", modelo)])


def grilla(tipo: str, cfg: dict) -> list[dict]:
    """Combinaciones de hiperparámetros a probar (config/reglas_negocio.yaml -> modelado.grillas)."""
    from itertools import product
    g = cfg["modelado"]["grillas"][tipo]
    return [dict(zip(g, valores)) for valores in product(*g.values())]


def ajustar_temporal(tipo: str, cfg: dict, Xe, ye, Xv, yv, variables: dict) -> pd.DataFrame:
    """Prueba cada combinación entrenando con ENTRENAMIENTO y midiendo en VALIDACION (2026-1)."""
    filas = []
    for params in grilla(tipo, cfg):
        modelo = construir_pipeline(tipo, params, variables, cfg["modelado"]["semilla"]).fit(Xe, ye)
        m = metricas(yv, modelo.predict_proba(Xv)[:, 1])
        filas.append({"tipo": tipo, **{f"param_{k}": v for k, v in params.items()},
                      "log_loss_validacion": m["log_loss"], "auc_validacion": m["auc_roc"],
                      "brier_validacion": m["brier"], "error_pct_validacion": m["error_pct"], "parametros": params})
    return pd.DataFrame(filas).sort_values("log_loss_validacion").reset_index(drop=True)


def intervalo_bootstrap(y, p, funcion, n: int = 1000, semilla: int = 7, umbral: float = 0.5) -> tuple[float, float]:
    """IC 95% por remuestreo: se repite la medición en 1.000 muestras con reemplazo del conjunto de prueba."""
    y, p = np.asarray(y, dtype=int), np.asarray(p, dtype=float)
    rng = np.random.default_rng(semilla)
    valores = []
    for _ in range(n):
        i = rng.integers(0, len(y), len(y))
        if len(np.unique(y[i])) < 2:
            continue
        valores.append(funcion(y[i], p[i], umbral))
    return float(np.percentile(valores, 2.5)), float(np.percentile(valores, 97.5))


def diferencia_error(y, p_antes, p_despues, n: int = 1000, semilla: int = 7, umbral: float = 0.5) -> tuple[float, float]:
    """IC 95% de (error del modelo nuevo − error del anterior), en puntos, remuestreando los MISMOS inscritos.
    Si el intervalo incluye el 0, la diferencia puede ser azar: los dos modelos rinden igual."""
    y = np.asarray(y, dtype=int)
    falla_a = (np.asarray(p_antes) >= umbral) != y
    falla_b = (np.asarray(p_despues) >= umbral) != y
    rng = np.random.default_rng(semilla)
    dif = [100 * (falla_b[i].mean() - falla_a[i].mean()) for i in (rng.integers(0, len(y), len(y)) for _ in range(n))]
    return float(np.percentile(dif, 2.5)), float(np.percentile(dif, 97.5))


def frase_docente(error_antes: float, error_despues: float, nombre: str, ic: tuple[float, float], n: int,
                  antes: str = "La regresión logística de la Semana 2", cuando: str = "inscritos de 2026-2") -> str:
    """La frase que pide el docente ("tenía un error del X%, este modelo del Y%"), sin afirmar una mejora que no hay."""
    num = lambda v: f"{v:.1f}".replace(".", ",")   # noqa: E731
    frase = (f"{antes} tenía un error del {num(error_antes)}%, mientras que el modelo de la Semana 3 ({nombre}) tiene "
             f"un error del {num(error_despues)}% en {n:,} {cuando}, que ningún modelo vio al entrenar.".replace(f"{n:,}", f"{n:,}".replace(",", ".")))
    dif = error_despues - error_antes
    if ic[1] < 0:
        return frase + f" Mejora {num(-dif)} puntos (IC 95% de la diferencia: {num(ic[0])} a {num(ic[1])})."
    if ic[0] > 0:
        return frase + f" Es {num(dif)} puntos peor (IC 95% de la diferencia: {num(ic[0])} a {num(ic[1])})."
    return (frase + f" La diferencia ({num(dif)} puntos) está dentro del margen del azar (IC 95%: {num(ic[0])} a "
            f"{num(ic[1])}): los dos modelos rinden igual.")


def nombres_salida(modelo) -> list[str]:
    """Nombres de las columnas que realmente ve el modelo (después del one-hot)."""
    return list(modelo.named_steps["preparacion"].get_feature_names_out())


In [ ]:
%%writefile flujo.py
"""Paso común de los sprints 3 a 6: cargar las tablas planas y construir las características del Sprint 2.

Así todos los sprints usan exactamente las mismas columnas, aprendidas solo con el entrenamiento.
"""
from __future__ import annotations

import pandas as pd

import caracteristicas as ca
import datos


def cargar_y_construir(fuente: str = "auto") -> dict:
    """Devuelve los inscritos con características, las referencias del Sprint 2 y el origen de los datos."""
    inscritos, origen = datos.cargar_inscritos(fuente)
    ref = ca.ajustar_referencias(inscritos)
    return {"inscritos": ca.construir_caracteristicas(inscritos, ref), "referencias": ref, "origen": origen}


def conjunto(d: pd.DataFrame, *nombres: str) -> pd.DataFrame:
    """Filtra por partición temporal: conjunto(d, 'ENTRENAMIENTO', 'VALIDACION')."""
    return d[d["particion"].isin(nombres)]


def modelos_finales(f: dict, parametros: dict) -> dict:
    """Reentrena los modelos finales del Sprint 3 con los hiperparámetros ya elegidos (los guarda modelos/*.json).

    Sirve en Google Colab, donde no están los archivos .joblib: con la misma semilla el resultado es idéntico.
    parametros = {"matricula": {"tipo", "parametros"}, "xgboost": {"parametros"}}
    """
    import modelado as mo
    cfg = ca.configuracion()
    semilla = cfg["modelado"]["semilla"]
    d, ref = f["inscritos"], f["referencias"]
    hist = conjunto(d, "ENTRENAMIENTO", "VALIDACION", "PRUEBA")
    Xh, yh = mo.preparar_X(hist, ref, mo.VARIABLES_MATRICULA), hist["matriculado"].astype(int)
    pm = parametros["matricula"]
    return {
        "referencias": ref,
        "modelo_matricula": mo.construir_pipeline(pm["tipo"], pm["parametros"], mo.VARIABLES_MATRICULA, semilla).fit(Xh, yh),
        "modelo_matricula_xgboost": mo.construir_pipeline("xgboost", parametros["xgboost"]["parametros"],
                                                          mo.VARIABLES_MATRICULA, semilla).fit(Xh, yh),
        # La regresión logística del Sprint 2 (sin ajuste de hiperparámetros), para compararla en igualdad de condiciones
        "logistica_sprint2": mo.construir_pipeline("logistica", cfg["modelado"]["sprint2_logistica"],
                                                   mo.VARIABLES_MATRICULA, semilla).fit(Xh, yh),
        "fondo_matricula": Xh.sample(300, random_state=1),
    }


In [ ]:
import sys
import warnings

sys.path.insert(0, ".")
warnings.filterwarnings("ignore")
EN_COLAB = True

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", "{:,.3f}".format)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
AZUL, NARANJA, GRIS = "#1B365D", "#E07B39", "#9AA5B1"

In [ ]:
import caracteristicas as ca
import flujo
import modelado as mo

# Lee las tablas planas por la API de Supabase (solo lectura): dataset_matricula y la rúbrica de la entrevista
# (dataset_entrevista). Después construye las columnas del Sprint 2.
f = flujo.cargar_y_construir("supabase" if EN_COLAB else "auto")
d, ref, cfg = f["inscritos"], f["referencias"], ca.configuracion()
print("Origen de los datos:", f["origen"])
print(f"Inscritos: {len(d):,} · con entrevista: {int(d['entrevista_realizada'].sum()):,}")
print(d["particion"].value_counts().to_string())

## 1. Variables de entrada
Solo se usan datos que existen **antes** de la decisión que se predice. Por ejemplo, los documentos entregados o la
forma de financiación no entran: se conocen al matricularse y serían fuga de datos.
Se agregan dos variables simples: `icfes_al_cuadrado`, para que la regresión logística pueda captar la "U invertida"
del Saber 11, y `semestre_2`, el ingreso a mitad de año. La entrevista entra como `indice_vocacional` (0 a 100).

In [ ]:
V = mo.VARIABLES_MATRICULA
display(pd.DataFrame([(tipo, ", ".join(cols)) for tipo, cols in V.items()], columns=["tipo", "variables (matrícula)"]))
y = lambda x: x["matriculado"].astype(int)
X = lambda x: mo.preparar_X(x, ref, V)
e, v, t = (flujo.conjunto(d, c) for c in ["ENTRENAMIENTO", "VALIDACION", "PRUEBA"])
ev = flujo.conjunto(d, "ENTRENAMIENTO", "VALIDACION")
print(f"Entrenamiento {len(e)} · validación {len(v)} · prueba {len(t)} inscritos")

## 2. Ajuste de hiperparámetros con validación temporal
Cada modelo tiene "perillas" (hiperparámetros): la regularización C de la regresión logística, la profundidad de los
árboles en Random Forest, la tasa de aprendizaje en XGBoost. Cada combinación se **entrena con 2024-1 a 2025-2 y se
mide en 2026-1**. Se elige la de menor **log-loss**, que castiga más las probabilidades seguras que salen mal.
Si un modelo más simple queda a menos de 0,002 del mejor, se prefiere el simple (navaja de Occam).

In [ ]:
COMPLEJIDAD = ["logistica", "random_forest", "xgboost"]
busquedas = {tipo: mo.ajustar_temporal(tipo, cfg, X(e), y(e), X(v), y(v), V) for tipo in COMPLEJIDAD}
for tipo, b in busquedas.items():
    print(f"\n{tipo}: {len(b)} combinaciones probadas; las 3 mejores en validación 2026-1")
    display(b.drop(columns="parametros").head(3).round(4))
mejores = {tipo: b.iloc[0] for tipo, b in busquedas.items()}
minimo = min(m["log_loss_validacion"] for m in mejores.values())
elegido = next(tp for tp in COMPLEJIDAD if mejores[tp]["log_loss_validacion"] <= minimo + 0.002)
print(f"\nModelo elegido por validación: {elegido} {mejores[elegido]['parametros']}")

**Lectura.** Los tres modelos quedan muy cerca. La regresión logística gana o empata: la relación entre las variables
y la matrícula es casi lineal en log-odds, y en ese caso los árboles no tienen patrones complejos que aprovechar y
corren más riesgo de sobreajustarse. Ojo: la matrícula de estos datos se simuló con un modelo logístico (supuesto
S-15), así que la logística tiene ventaja por construcción; con datos reales la comparación puede cambiar.

## 3. Comparación final en la prueba 2026-2
Cada modelo se reentrena con entrenamiento + validación y se evalúa **una sola vez** en 2026-2, un periodo que no se
usó para elegir nada. La logística del Sprint 2 (C = 1, sin ajuste) también se reentrena así, para compararla en
igualdad de condiciones.
- **Error %:** inscritos mal clasificados con umbral de 50%.
- **AUC:** probabilidad de que el modelo ponga a un matriculado por encima de un no matriculado; 0,5 es azar.
- **Brier:** error cuadrático de la probabilidad; más bajo es mejor.

In [ ]:
semilla = cfg["modelado"]["semilla"]
probs = {"Regresión logística (Sprint 2)": mo.construir_pipeline("logistica", cfg["modelado"]["sprint2_logistica"], V, semilla)
                                             .fit(X(ev), y(ev)).predict_proba(X(t))[:, 1]}
NOMBRES = {"logistica": "Regresión logística ajustada", "random_forest": "Random Forest", "xgboost": "XGBoost"}
for tipo in COMPLEJIDAD:
    modelo = mo.construir_pipeline(tipo, mejores[tipo]["parametros"], V, semilla).fit(X(ev), y(ev))
    probs[NOMBRES[tipo]] = modelo.predict_proba(X(t))[:, 1]
res = {n: mo.metricas(y(t), q) for n, q in probs.items()}
comparacion = mo.tabla_metricas(res)
display(comparacion[["n", "error_pct", "auc_roc", "auc_pr", "brier"]])
b, m = res["Regresión logística (Sprint 2)"], res[NOMBRES[elegido]]
dif = mo.diferencia_error(y(t), probs["Regresión logística (Sprint 2)"], probs[NOMBRES[elegido]])
print("\n" + mo.frase_docente(b["error_pct"], m["error_pct"], NOMBRES[elegido], dif, len(t)))
from sklearn.metrics import roc_auc_score
ic = mo.intervalo_bootstrap(y(t), probs[NOMBRES[elegido]], lambda a, q, u: roc_auc_score(a, q))
print(f"AUC del modelo elegido: {m['auc_roc']:.3f} (IC 95% por bootstrap: {ic[0]:.3f} a {ic[1]:.3f})")

In [ ]:
from sklearn.calibration import calibration_curve
from sklearn.metrics import confusion_matrix, roc_curve
fig, ejes = plt.subplots(1, 3, figsize=(17, 4.5))
for n, q in probs.items():
    fpr, tpr, _ = roc_curve(y(t), q); ejes[0].plot(fpr, tpr, label=f"{n} ({res[n]['auc_roc']:.3f})")
    fr, me = calibration_curve(y(t), q, n_bins=8, strategy="quantile"); ejes[1].plot(me, fr, "o-", label=n)
ejes[0].plot([0, 1], [0, 1], ":", color=GRIS); ejes[0].set(title="Curva ROC (prueba 2026-2)", xlabel="falsos positivos", ylabel="verdaderos positivos")
ejes[1].plot([0, 1], [0, 1], ":", color=GRIS); ejes[1].set(title="Calibración", xlabel="probabilidad predicha", ylabel="tasa real")
for ax in ejes[:2]:
    ax.legend(fontsize=7)
cm = confusion_matrix(y(t), probs[NOMBRES[elegido]] >= 0.5)
ejes[2].imshow(cm, cmap="Blues")
for (i, j), val in np.ndenumerate(cm):
    ejes[2].text(j, i, val, ha="center", va="center", fontsize=13)
ejes[2].set_xticks([0, 1], ["predice NO", "predice SÍ"]); ejes[2].set_yticks([0, 1], ["no se matriculó", "se matriculó"])
ejes[2].set_title(f"Matriz de confusión · {NOMBRES[elegido]}")
plt.tight_layout(); plt.show()

**Lectura.**
- El elegido **no mejora** a la logística del Sprint 2: la
  diferencia cabe en el margen del azar (con 761 inscritos, unos pocos casos). Random Forest y XGBoost tampoco la
  superan. Se reporta tal cual: el modelo se eligió con la validación, antes de mirar la prueba, y no se cambia después.
- Todos se equivocan en cerca de 4 de cada 10 inscritos. La decisión de matricularse depende de factores que no están
  en los datos (familia, otras ofertas, trabajo).
- Su valor está en **ordenar**: a los de mayor probabilidad se les asegura la matrícula y a los indecisos se les hace
  seguimiento (Sprint 5).
- Las curvas de calibración cerca de la diagonal indican que un 60% predicho significa cerca de 60% real.

## 4. Modelos finales y tabla de predicciones (bloque 3 del tablero)
El modelo elegido se reentrena con **todo el histórico** (2024-1 a 2026-2) para predecir la campaña 2027-1. La tabla
`prediccion` (también en Supabase) tiene, por inscrito, la probabilidad de matrícula, el decil y el segmento de
propensión.

In [ ]:
hist = flujo.conjunto(d, "ENTRENAMIENTO", "VALIDACION", "PRUEBA")
final = mo.construir_pipeline(elegido, mejores[elegido]["parametros"], V, semilla).fit(X(hist), y(hist))
pm = final.predict_proba(X(d))[:, 1].round(4)
s = cfg["modelado"]["segmentos_propension"]
prediccion = pd.DataFrame({"inscripcion_id": d["inscripcion_id"], "periodo": d["periodo"], "prob_matricula": pm,
                           "segmento_propension": np.select([pm >= s["ALTA"], pm < s["BAJA"]], ["ALTA", "BAJA"], "MEDIA")})
display(prediccion[prediccion["periodo"] == "2027-1"].sort_values("prob_matricula", ascending=False).head(10))
print(prediccion.groupby("periodo")["segmento_propension"].value_counts().unstack().fillna(0).astype(int))

## Conclusiones del Sprint 3
1. El modelo de Machine Learning **ya predice** la matrícula de cada inscrito y ordena mucho mejor que el azar (AUC
   cercano a 0,66).
2. La regresión logística queda por delante de Random Forest y XGBoost: más simple, igual o más exacta y más fácil de
   explicar. Se elige con evidencia, no por preferencia.
3. Ajustar sus hiperparámetros no mejora a la logística del Sprint 2: los dos rinden igual dentro del margen del azar.
   Con estos datos, la mejora no viene de un modelo más complejo sino de mejores datos (por ejemplo, el histórico
   real de Admisiones).
4. Los modelos finales quedan guardados (`sprints/sprint_3_modelado/modelos/`) y los usan los sprints 4 a 6.